# einops-reduce composite — cx8: per-batch mean then re-tile across a new heads axis

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `einops-reduce`, `einops-repeat`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "einops-reduce"
DD_ATOM_IDS = ["einops-reduce", "einops-repeat"]
DD_SUBTOPICS = ["Einops: Reduce", "Einops: Repeat"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

Reduce and repeat are inverses on the named-axis side: `reduce` drops a labelled axis, `repeat` introduces a new labelled axis bound by a kwarg. Composing them in one fn — `reduce` collapses the existing C axis to a scalar-per-batch, then `repeat` materialises a fresh `heads` axis of size H so every head sees the same per-batch summary — is the standard ARENA move when you need a broadcasted query for each head/sample/copy.

This is NOT the same as broadcasting back into the original C axis. `repeat` actually introduces a NEW named output dim, with `heads=H` as a kwarg binding.

### Composite Exercise — per-batch mean then re-tile across a new heads axis

**Atoms exercised together**: `einops-reduce`, `einops-repeat`

Implement `cx8_per_batch_mean_per_head(x, heads)` that takes a tensor of shape `(B, C)` and an `int heads` and returns a tensor of shape `(B, heads)` where every head row is a copy of the per-batch mean over C.

Two atoms in one expression:

1. **Reduce** the channel axis with `einops.reduce(x, 'b c -> b', 'mean')` — note the axis is fully removed (no size-1 slot here, because the next step adds a *different* named axis).
2. **Repeat** the per-batch scalar across a fresh `heads` axis with `einops.repeat(per_batch, 'b -> b heads', heads=heads)`. The new axis is bound by kwarg, not derived from the input.

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx8_per_batch_mean_per_head(x, heads):
    raise NotImplementedError

def _test_cx8():
    x = t.tensor([[1.0, 3.0, 5.0], [2.0, 4.0, 6.0]])  # (B=2, C=3)
    out = cx8_per_batch_mean_per_head(x, heads=4)
    assert out.shape == (2, 4), f'expected (2,4), got {out.shape}'
    # Every head row of a given batch must equal the per-batch mean.
    expected_per_batch = t.tensor([3.0, 4.0])
    for h in range(4):
        assert t.allclose(out[:, h], expected_per_batch), out

    # Case B: random + heads=1 (degenerate but legal).
    x2 = t.randn(5, 7)
    out2 = cx8_per_batch_mean_per_head(x2, heads=1)
    assert out2.shape == (5, 1)
    assert t.allclose(out2[:, 0], x2.mean(dim=1))

    # Case C: heads=8 — verify stride-0 / repeat semantics (all heads identical).
    x3 = t.randn(3, 4)
    out3 = cx8_per_batch_mean_per_head(x3, heads=8)
    assert out3.shape == (3, 8)
    assert t.allclose(out3.std(dim=1), t.zeros(3), atol=1e-6), 'heads axis should be uniform'
    _dd_passed.add('cx8')

_test_cx8()

<details><summary>Show solution — cx8</summary>

```python
def cx8_per_batch_mean_per_head(x, heads):
    # Atom 1: reduce — drop the C axis entirely (b c -> b).
    per_batch = reduce(x, 'b c -> b', 'mean')
    # Atom 2: repeat — introduce a NEW named axis 'heads', bound by kwarg.
    return repeat(per_batch, 'b -> b heads', heads=heads)
```

The reduce drops a labelled axis; the repeat introduces one. They are not the same axis — that would be no-op identity. The kwarg `heads=heads` is what binds the new axis size.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx8'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx8',
        'subtopics': ["Einops: Reduce", "Einops: Repeat"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()